# 실전 프로젝트 주제 선정

## 이번 프로젝트의 목표
1. 스타벅스 고객의 프로모션 반응과 실제 구매 전환을 분석해서, 다음 캠페인 전략에 쓸 수 있는 Tableau 대시보드를 만드는 것.
2. 오퍼 발송 → 열람 → 완료 → 구매 흐름을 고객 세그먼트와 함께 보는 CRM 퍼널 분석
    - CRMM?: 어떤 고객이 반응하고, 어떤 고객이 실제 구매까지 이어지는가
3. 고객군과 오퍼 유형별 성과를 비교해 다음 프로모션 타겟팅 전략을 결정할 수 있게 한다.

## 확인된 컬럼 특징

- **profile.csv**
    - 크기: 17,000행 × 5열
    - 역할: 고객의 인구통계 및 가입 정보를 담고 있는 테이블
    - 주요 컬럼
        - gender : 고객 성별
        - age : 고객 나이
        - id : 고객 고유 ID
        - became_member_on : 멤버십 가입일(YYYYMMDD 형식)
        - income : 고객 연간 소득(달러)

- 확인한 특징: 
    - gender 결측: 2,175건
    - income 결측: 2,175건
    - age=118 값이 2,175건

고객 세그먼트 분석을 하려면 age=118 처리 기준과 결측치 처리를 팀에서 먼저 정해야 한다

- **portfolio**
    - 크기: 10행 × 6열
    - 역할: 프로모션(오퍼) 자체의 속성을 담고 있는 테이블
    - 주요 컬럼
        - reward : 프로모션 완료 시 고객에게 주는 보상 금액
        - channels : 프로모션이 전달된 채널 목록(web, email, mobile, social)
        - difficulty : 보상을 받기 위해 필요한 최소 지출 금액
        - duration : 프로모션 유효 기간(일 단위)
        - offer_type : 프로모션 유형(bogo, discount, informational)
        - id : 프로모션 고유 ID

- 확인한 특징:
    - offer_type
        - bogo 4개
        - discount 4개
        - informational 2개
    - channels는 리스트 구조다.
        - 예: ['web', 'email', 'mobile', 'social']

channels는 파싱해서 써야 한다.

- **transcript**
    - 크기: 306,534행 × 4열
    - 고객 수: 17,000명
    - 역할: 고객의 프로모션 반응 및 거래 로그를 기록한 이벤트 테이블
    - 주요 컬럼
        - person : 고객 ID
        - event : 이벤트 유형(transaction, offer received, offer viewed, offer completed)
        - value : 이벤트별 세부 정보
        - time : 시간 정보 (데이터 내 상대적 시간, t=0부터 시작)

- 이벤트 분포는 다음과 같다
    - transaction: 138,953
    - offer received: 76,277
    - offer viewed: 57,725
    - offer completed: 33,579

- 확인한 특징:
    - value는 딕셔너리 문자열 구조다. 
        - 어떤 행은 {'offer id': '...'} 형태
        - 어떤 행은 {'offer_id': '...', 'reward': ...} 형태
        - 어떤 행은 {'amount': ...} 형태
        - 이벤트별로 value 안에서 꺼내야 하는 키가 다르다.

transcript.person(고객 ID)가 profile.id와 매칭되는지 확인이 필요\
transcript.offer_id(오퍼 ID)가 portfolio.id와 매칭되는지 확인이 필요

- starbucks_menu_260112.csv
    - 제품 영양정보 테이블
    - 행 수: 195
    - 컬럼: 제품명, 칼로리, 당류, 카페인, 나트륨 등

핵심 CRM 분석용 메인 테이블이 아님\
이번 분석에서 중요하게 사용되지 않을 가능성이 높음

## 만약 분석을 한다면 무엇을 볼것인가?

주요축 간단 설명

- 고객:\
프로모션을 받는 대상인 스타벅스 회원\
성별, 연령, 소득, 가입 시점 같은 특성으로 구분해 볼 수 있다.

- 오퍼:\
스타벅스가 고객에게 보내는 프로모션 제안\
할인, 1+1, 혜택 안내, 정보성 안내 등

- 채널:\
오퍼가 고객에게 전달되는 경로이다.\
web, email, mobile, social 등의 채널 조합

- 퍼널:\
고객이 오퍼를 받은 뒤 열람, 완료, 구매까지 이어지는 단계별 흐름\
각 단계에서 얼마나 반응하고 이탈하는지 파악 가능

**매출이 얼마 나왔나**가 아닌\
어떤 고객(고객 세그먼트별)이 어떤 프로모션에 반응하고, 그 반응이 실제 구매로 얼마나 이어지는가?

> 이때의 구매전환은 회의를 통해 결정할 필요가 있음\
오퍼를 받은 뒤 거래가 있으면 구매 전환인가?\
오퍼를 본 뒤 거래가 있으면 구매 전환인가?\
completed 이후 거래인가? 등

1. 고객 세그먼트별
    - 성별, 연령, 소득, 가입 시점에 따라 반응 차이가 있는가?
2. 오퍼별
    - bogo, discount, informational 중 어떤 유형이 잘 작동하는가?
    - reward, difficulty, duration 차이가 영향을 주는가?
3. 채널별
    - web, email, mobile, social의 포함 여부 및 조합에 따라 반응률 차이가 있는가?
4. 퍼널별
    - 오퍼를 받은 뒤 본 비율
    - 오퍼를 본 뒤 완료한 비율
    - 오퍼 이후 실제 구매가 얼마나 일어나는가?

## 그래서 설정한 분석 목표는??
**고객 세그먼트별 프로모션 반응과 구매 전환 분석**

- 이 분석 목표의 장점:
    1. 주어진 테이블(profile, portfolio, transcript)을 모두 활용할 수 있음
    2. 이를 통해 퍼널분석이 가능함

- 이 분석 목표의 단점:
    1. 핵심 테이블을 사용하는 만큼 데이터 정제 및 전처리가 복잡해진다.

## 최종 목표

고객 특성과 오퍼 특성을 함께 고려했을 때,\
어떤 고객군에서 어떤 프로모션이 **더 높은 반응과 구매 전환을 만드는지 파악**한다.
> 스타벅스 고객 세그먼트별 프로모션 반응과 구매 전환은 어떻게 다른가?

1. 캠페인 퍼널 구조 파악
    - 오퍼 수신, 열람, 완료, 구매 흐름을 수치화
    - 어느 단계에서 이탈이 큰지 확인
2. 고객 특성별 반응 차이 분석
    - 성별, 연령대, 소득대, 가입 시점별 반응률 비교
    - 누가 오퍼를 더 잘 보고, 잘 완료하고, 많이 구매하는지 확인
3. 오퍼 유형별 성과 비교
    - bogo, discount, informational 비교
    - reward, difficulty, duration, channels에 따른 성과 차이 확인
4. 고객군 × 오퍼 조합 인사이트
    - 어떤 고객군에 어떤 오퍼가 잘 맞는지 제안
    - 추후 시행할 캠페인의 타겟 및 전략 제시

- 대시보드 사용자 및 페르소나 설정
    - 마케팅 실무자용 대시보드

- 핵심 KPI 선정
    - 총 고객수
    - 오퍼 발송수
    - 오퍼 열람수, 열람률
    - 오퍼 완료수, 완료율
    - 거래 건수
    - 총 거래 금액
    - 고객당 평균 거래 금액

- 계산? 전환? 파싱? KPI
    - 계산식 써서 만드는 KPI
    - 구매 전환율?

설무아 튜터님 팁

became_member_on을 그대로 보지 말고 파생변수로 만드는것이 좋다.\
가입 연도, 가입 후 경과 기간, 신규 / 기존 고객 구분 등으로 나누는것이 가능함.